# Guided Lab: End-to-End RAG Pipeline — Bramblewood Gear Co. / FieldWise

**Company:** Bramblewood Gear Co., an outdoor equipment retailer.
**Product:** FieldWise, the internal assistant support agents use while
working a ticket.

This lab builds a complete retrieval-augmented generation workflow around
one real document: Bramblewood's own *Product Care & Warranty Handbook*.

What you will do:
- generate the handbook as a real multi-page PDF
- load and split it
- create embeddings (with an offline-safe fallback)
- store chunks in ChromaDB
- answer questions with Groq (with an offline-safe fallback)
- trace the run with LangSmith (with an offline-safe fallback)

## Why this document?

A real company handbook gives FieldWise enough material for meaningful
retrieval: several product categories, troubleshooting steps, and a policy
section, all cross-referenced the way a real support document would be.

Rather than pulling a handbook from Bramblewood's intranet (which this
notebook obviously can't reach), we generate the same document a support-ops
team would actually produce -- the kind of PDF you'd get by exporting an
internal wiki page to PDF -- and then run the exact same load -> split ->
embed -> store -> retrieve -> generate pipeline against it. Everything after
Section 3 works identically whether the PDF came from a real intranet
export or from this notebook's generator.

## 1) Install packages

In [ ]:
%pip install -qU python-dotenv reportlab pypdf langchain langchain-community langchain-text-splitters langchain-chroma chromadb langchain-groq scikit-learn

## 2) Configure environment variables

Create a `.env` file with:

```env
GROQ_API_KEY=your_groq_key
GROQ_MODEL=llama-3.3-70b-versatile
LANGSMITH_API_KEY=your_langsmith_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=fieldwise-rag-pipeline
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "fieldwise-rag-pipeline")

print("GROQ_API_KEY set:", bool(GROQ_API_KEY))
print("GROQ_MODEL:", GROQ_MODEL)
print("LANGSMITH_API_KEY set:", bool(LANGSMITH_API_KEY))
print("LANGSMITH_PROJECT:", LANGSMITH_PROJECT)

## 3) Build the handbook PDF

Six product sections plus a returns-and-warranty policy section, generated
as a real PDF with `reportlab` so the rest of this notebook can load it with
an ordinary PDF loader, exactly like any other document FieldWise ingests.

In [ ]:
from pathlib import Path
from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch

DATA_DIR = Path("./fieldwise_pipeline_data")
DATA_DIR.mkdir(exist_ok=True)
PDF_PATH = DATA_DIR / "bramblewood_handbook.pdf"


def build_handbook_pdf(path: Path) -> Path:
    if path.exists() and path.stat().st_size > 0:
        print(f"Using existing handbook: {path}")
        return path

    styles = getSampleStyleSheet()
    doc = SimpleDocTemplate(str(path), pagesize=LETTER, topMargin=0.9 * inch, bottomMargin=0.9 * inch)
    story = []

    def heading(text):
        story.append(Paragraph(text, styles["Heading1"]))
        story.append(Spacer(1, 8))

    def body(text):
        story.append(Paragraph(text, styles["BodyText"]))
        story.append(Spacer(1, 10))

    story.append(Paragraph("Bramblewood Gear Co.", styles["Title"]))
    story.append(Paragraph("Product Care &amp; Warranty Handbook", styles["Title"]))
    story.append(Spacer(1, 20))
    body(
        "This handbook collects gear-care instructions, troubleshooting steps, "
        "and warranty terms for Bramblewood Gear Co. products. It is the "
        "reference support agents use when answering tickets through "
        "FieldWise, Bramblewood's internal support assistant, and the source "
        "document for this notebook's retrieval pipeline."
    )
    story.append(PageBreak())

    heading("1. Tents")
    body(
        "Summit Ridge 2P Tent: a 3-season, freestanding tent with aluminum "
        "poles and a silicone-coated rainfly, built for three-season "
        "backpacking trips."
    )
    body(
        "Alpine Trail 4P Tent: a 4-season expedition tent with a "
        "HydroShield-coated rainfly and reinforced poles rated for high "
        "wind and snow load."
    )
    body(
        "Both tents ship with a factory-applied waterproof coating on the "
        "rainfly that gradually wears down with UV exposure. When the "
        "coating wears down, water can begin to seep through the seams and "
        "fabric panels during sustained rain. The fix is straightforward: "
        "reapply a DWR (durable water repellent) treatment to the rainfly "
        "once a year, and reseal the floor and rainfly seams with seam "
        "sealer. If leaking continues after reproofing and the tent is less "
        "than two years old, treat it as a hard-goods warranty claim rather "
        "than a general return."
    )
    body(
        "Aluminum tent poles can develop hairline cracks near the ferrules "
        "(the metal sleeves that join pole sections) after repeated setup in "
        "high wind. A cracked pole should be repaired with a pole splint "
        "before the next trip; a snapped pole within the warranty period is "
        "replaced at no cost."
    )
    story.append(PageBreak())

    heading("2. Sleeping Bags")
    body(
        "Ember Down 20F Sleeping Bag: fills with 650-fill DriDown, a "
        "water-resistant treated goose down rated to 20 degrees Fahrenheit."
    )
    body(
        "Down loses loft, and warmth, when it's washed with regular "
        "detergent, which strips the natural oils and the DriDown "
        "treatment. Wash only with a technical down soap on a gentle cycle "
        "in a front-loading machine. Tumble dry on low heat with two or "
        "three clean tennis balls or wool dryer balls to break up clumps as "
        "the down dries; this can take several cycles. Store the bag "
        "loosely in its large mesh storage sack, never compressed in its "
        "stuff sack, since long-term compression is the single biggest "
        "cause of premature loft loss."
    )
    story.append(PageBreak())

    heading("3. Backpacks")
    body(
        "Basecamp 65L Backpack: an internal-frame pack with an adjustable "
        "torso length (S/M/L) and a load-bearing hip belt, intended for "
        "multi-day trips."
    )
    body(
        "The main compartment's zipper slider is the most common point of "
        "failure after heavy use. A stiff or sticking slider can usually be "
        "freed with a dry silicone lubricant applied along the zipper teeth. "
        "A slider that has fully separated from the zipper tape, or teeth "
        "that no longer close, is a workmanship defect covered under "
        "Basecamp's lifetime warranty, and can be repaired or replaced at no "
        "cost regardless of how long the customer has owned it."
    )
    story.append(PageBreak())

    heading("4. Stoves")
    body(
        "TrailFlame Compact Stove: a canister-fed backpacking stove with a "
        "piezo (push-button spark) ignition."
    )
    body(
        "The most common support issue is a piezo igniter that stops "
        "sparking after repeated use in wet or cold conditions. This does "
        "not mean the stove is unusable: any lighter or match can ignite the "
        "burner while the igniter is unavailable. A piezo igniter that has "
        "stopped working within two years of purchase is covered under the "
        "standard hard-goods warranty and can be replaced as a standalone "
        "part. Fuel canisters from other brands using a standard threaded "
        "valve are compatible, but Bramblewood only warranties performance "
        "with its own TrailFlame canisters."
    )
    story.append(PageBreak())

    heading("5. Footwear")
    body(
        "Ridgeline Hiking Boots: leather and mesh uppers over a waterproof "
        "membrane, with a resoleable Vibram outsole."
    )
    body(
        "The waterproof membrane typically holds up for a full season of "
        "regular use before it needs reproofing; the first sign is water "
        "darkening the upper fabric instead of beading off. An annual "
        "spray-on DWR reproofing treatment restores this. Once the outsole "
        "tread wears down, boots can be sent through Bramblewood's resole "
        "program instead of being replaced outright, which is available for "
        "the life of the boot for a flat fee, separate from the one-year "
        "manufacturing warranty."
    )
    story.append(PageBreak())

    heading("6. Returns &amp; Warranty Policy")
    body(
        "Returns: unused gear in its original packaging can be returned "
        "within 60 days of purchase for a full refund. Gear that has been "
        "used or worn, including a tent that has only been pitched once, is "
        "not eligible for a general return."
    )
    body(
        "Warranty: hard goods (tents, stoves) carry a 2-year warranty "
        "against manufacturing defects. Basecamp backpacks carry a lifetime "
        "warranty against defects in materials and workmanship under normal "
        "use. Ridgeline boots carry a 1-year warranty against manufacturing "
        "defects, separate from the paid resole program described above."
    )
    body(
        "Any used item reported as defective should be filed as a warranty "
        "claim, not a return, regardless of how long it's been owned, as "
        "long as the issue matches a covered defect type for that product "
        "category."
    )

    doc.build(story)
    print(f"Built: {path}")
    return path


build_handbook_pdf(PDF_PATH)

## 4) Load the PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print("Pages loaded:", len(pages))
print("First page metadata:", pages[0].metadata)
print("First page preview:")
print(pages[0].page_content[:500])

## 5) Split into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
)

chunks = splitter.split_documents(pages)

print("Chunk count:", len(chunks))
print("Example chunk metadata:", chunks[0].metadata)
print("Example chunk preview:")
print(chunks[0].page_content[:400])

## 6) Embedding availability check

FieldWise's production deployment uses a local Hugging Face
sentence-transformer for embeddings. We check for it once and fall back to
a deterministic offline embedding if it isn't reachable (no package, or no
network access to huggingface.co), so the rest of the pipeline runs the
same way either way.

In [ ]:
from langchain_core.embeddings import Embeddings
from sklearn.feature_extraction.text import HashingVectorizer


class OfflineHashingEmbeddings(Embeddings):
    """Deterministic, dependency-free embedding fallback. Uses the hashing
    trick, so it needs no fitted vocabulary and stays numerically consistent
    across separate processes without persisting any state -- handy later
    in this module, when a subprocess MCP server needs to re-open the same
    Chroma collection. It's a lexical fallback, not a semantic one."""

    def __init__(self, n_features: int = 512):
        self.n_features = n_features
        self._hv = HashingVectorizer(
            n_features=n_features, alternate_sign=False, norm="l2", stop_words="english"
        )

    def embed_documents(self, texts):
        return self._hv.transform(texts).toarray().tolist()

    def embed_query(self, text):
        return self._hv.transform([text]).toarray()[0].tolist()


def get_embeddings():
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
        candidate = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        candidate.embed_query("connectivity check")
        print("Embeddings: live sentence-transformers model loaded (semantic).")
        return candidate, "live"
    except Exception as e:
        print(f"Embeddings: falling back to offline hashing embeddings -- {type(e).__name__}: {str(e)[:160]}")
        return OfflineHashingEmbeddings(), "offline"


embeddings, EMBEDDINGS_MODE = get_embeddings()

## 7) Store chunks in ChromaDB

In [ ]:
from langchain_chroma import Chroma
import shutil

CHROMA_DIR = DATA_DIR / "chroma_store"
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="bramblewood_handbook",
    persist_directory=str(CHROMA_DIR),
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("Chroma directory:", CHROMA_DIR.resolve())

## 8) Inspect retrieval

In [ ]:
query = "What tent models are available and how do they differ?"
docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"--- Retrieved {i} ---")
    print("page:", doc.metadata.get("page"))
    print(doc.page_content[:400])
    print()

## 9) Groq availability check

We check once whether Groq is reachable and fall back to a small
deterministic mock chat model if not, so the LCEL chain below runs the same
way either way. On your own machine with `GROQ_API_KEY` set, this uses the
real model.

In [ ]:
import re
from typing import Any, List, Optional

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult


def _extract_section(text: str, label: str) -> str:
    """Pull a labeled section like 'Context:' out of a rendered prompt
    string, stopping at the next 'Label:' marker (or the end of the
    string) so it doesn't grab everything that follows."""
    pattern = rf"{re.escape(label)}:\s*(.*?)(?=\n\s*[A-Z][a-zA-Z ]{{0,20}}:|\Z)"
    match = re.search(pattern, text, flags=re.DOTALL)
    return match.group(1).strip() if match else ""


class OfflineMockChatModel(BaseChatModel):
    """Deterministic offline stand-in for ChatGroq: an extractive heuristic
    (pick the context sentence(s) sharing the most words with the question)
    so LCEL chains keep working end-to-end without a live Groq connection.
    Every response is prefixed with "[offline mock]"."""

    @property
    def _llm_type(self) -> str:
        return "offline-mock-chat-model"

    def _generate(self, messages: List[BaseMessage], stop: Optional[List[str]] = None, run_manager: Optional[Any] = None, **kwargs: Any) -> ChatResult:
        human_text = " ".join(m.content for m in messages if m.type == "human" and isinstance(m.content, str))
        context = _extract_section(human_text, "Context") or human_text
        question = _extract_section(human_text, "Question") or human_text
        answer = self._extractive_answer(question, context)
        message = AIMessage(content=f"[offline mock] {answer}")
        return ChatResult(generations=[ChatGeneration(message=message)])

    @staticmethod
    def _extractive_answer(question: str, context: str, max_sentences: int = 3) -> str:
        q_terms = {w.lower() for w in re.findall(r"[a-zA-Z]{4,}", question)}

        # Context arrives as several retrieved chunks separated by blank
        # lines (see format_docs / make_context). Score whole chunks first
        # and extract only from the best-matching one, rather than pooling
        # sentences across all chunks -- otherwise a stray keyword match in
        # an unrelated chunk can bleed into the answer.
        chunk_blocks = [c.strip() for c in re.split(r"\n\s*\n", context.strip()) if c.strip()]
        if not chunk_blocks:
            chunk_blocks = [context.strip()]

        def term_overlap(text: str) -> int:
            text_terms = {w.lower() for w in re.findall(r"[a-zA-Z]{4,}", text)}
            return len(q_terms & text_terms)

        best_chunk = max(chunk_blocks, key=term_overlap)

        sentences = [
            s for s in re.split(r"(?<=[.!?])\s+", best_chunk.strip())
            if len(s.strip()) > 15
        ]
        scored = []
        for s in sentences:
            if s.strip():
                scored.append((term_overlap(s), s.strip()))
        scored.sort(key=lambda x: x[0], reverse=True)
        top = [s for _, s in scored[:max_sentences]] or (sentences[:1] if sentences else [""])
        return " ".join(top)[:700]


def get_llm():
    try:
        from langchain_groq import ChatGroq
        candidate = ChatGroq(model=GROQ_MODEL, temperature=0)
        candidate.invoke("connectivity check")
        print("LLM: live Groq model connected.")
        return candidate, "live"
    except Exception as e:
        print(f"LLM: falling back to offline mock chat model -- {type(e).__name__}: {str(e)[:160]}")
        return OfflineMockChatModel(), "offline"


llm, LLM_MODE = get_llm()

## 10) Build the LCEL RAG chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        f"[page={doc.metadata.get('page')}] {doc.page_content}"
        for doc in docs
    )

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are FieldWise, Bramblewood's support assistant. Answer only "
        "from the provided handbook context. If the answer is not in the "
        "context, say you do not know."
    ),
    (
        "user",
        "Question: {question}\n\nContext:\n{context}\n\nAnswer in 4 sentences or fewer."
    ),
])

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("LCEL RAG chain ready. LLM mode:", LLM_MODE)

## 11) Ask questions

In [ ]:
questions = [
    "What tent models are available and how do they differ?",
    "How do I stop a tent rainfly from leaking?",
    "Is a broken zipper slider on a Basecamp backpack covered under warranty?",
]

for q in questions:
    print("QUESTION:", q)
    print("ANSWER:", rag_chain.invoke(q))
    print()

## 12) LangSmith tracing

We check once whether the LangSmith API is reachable and disable tracing
for this run if it isn't, so the notebook doesn't spend the rest of its
run retrying an unreachable endpoint in the background.

In [ ]:
def check_langsmith_tracing():
    if not LANGSMITH_API_KEY:
        print("LangSmith: no API key set -- tracing disabled for this run.")
        os.environ["LANGSMITH_TRACING"] = "false"
        return "disabled"
    try:
        from langsmith import Client
        Client().has_dataset(dataset_name="__connectivity_check__")
        print("LangSmith: API reachable, tracing enabled.")
        os.environ["LANGSMITH_TRACING"] = "true"
        return "live"
    except Exception as e:
        print(f"LangSmith: API not reachable -- {type(e).__name__}: {str(e)[:160]}. Disabling tracing for this run.")
        os.environ["LANGSMITH_TRACING"] = "false"
        return "offline"


LANGSMITH_MODE = check_langsmith_tracing()

from langsmith import traceable

@traceable(name="rag_pdf_question_answer")
def answer_question(question: str) -> str:
    return rag_chain.invoke(question)

print(answer_question("What does the handbook cover for stoves?"))

## 13) Helper that prints retrieval plus answer

In [ ]:
def ask(question: str):
    retrieved = retriever.invoke(question)

    print("Retrieved chunks:")
    for i, doc in enumerate(retrieved, 1):
        print(f"[{i}] page={doc.metadata.get('page')}")
        print(doc.page_content[:300].replace("\n", " "))
        print()

    answer = answer_question(question)
    print("Final answer:")
    print(answer)

ask("What footwear is available and how should it be cared for?")

## 14) Clean up

In [ ]:
def cleanup():
    if CHROMA_DIR.exists():
        shutil.rmtree(CHROMA_DIR)
        print("Removed", CHROMA_DIR)
    else:
        print("Nothing to remove")

# cleanup()

## Key takeaways

- A real handbook (generated here, exported from a wiki in production)
  makes the RAG lab meaningful -- the same load -> chunk -> embed -> Chroma
  -> LCEL RAG pipeline runs unchanged either way.
- Groq provides the generation layer, with a deterministic offline fallback
  so the notebook still runs end-to-end without a live connection.
- LangSmith tracing makes the pipeline observable when it's reachable, and
  is cleanly disabled (not just silently failing) when it isn't.
- ChromaDB gives FieldWise a persistent local index it can reopen later --
  this is exactly what the MCP notebook in this module does from a
  separate process.

## References

- LangChain RAG: https://docs.langchain.com/oss/python/langchain/rag
- LangChain knowledge base: https://docs.langchain.com/oss/python/langchain/knowledge-base
- Groq integration: https://docs.langchain.com/oss/python/integrations/providers/groq
- Chroma integration: https://docs.langchain.com/oss/python/integrations/providers/chroma
- Trace LangChain applications: https://docs.langchain.com/langsmith/trace-with-langchain

## Verification summary

| Section | Status | Notes |
|---|---|---|
| Package install | Executed | pip install cell |
| Handbook PDF generation + load + split | Executed & verified | real PDF on disk, loaded with `PyPDFLoader`, 7 pages |
| Embeddings | Executed & verified | see printed `EMBEDDINGS_MODE` -- offline hashing fallback in a sandbox with no Hugging Face access, live sentence-transformers on a machine with internet |
| Chroma storage + retrieval | Executed & verified | persisted to `./fieldwise_pipeline_data/chroma_store` |
| LCEL RAG chain + question answering | Executed & verified | see printed `LLM_MODE` -- offline mock in a sandbox with no Groq access, live Groq on a machine with `GROQ_API_KEY` set |
| LangSmith tracing | Executed & verified | see printed `LANGSMITH_MODE` -- disabled cleanly when the API isn't reachable, live when `LANGSMITH_API_KEY` is set and the endpoint is reachable |

On a machine with real `GROQ_API_KEY` and `LANGSMITH_API_KEY` values and
network access, `LLM_MODE` and `LANGSMITH_MODE` will both read `live`, and
`rag_chain` will return real generated answers with traces visible in the
LangSmith UI.